# Baseline GLM — Mitt

**Goal this week: a WORKING pipeline, not a good model.** This notebook is the first time data flows
all the way from raw `.npz` files to a trained model producing a real prediction. We're deliberately
using logistic regression (a GLM) instead of the RNN, for two reasons: it's simple enough to trust
immediately (no hidden bugs from a complex architecture), and it matches the meeting guidance to start
simple and keep interpretability front and center.

If this pipeline works end to end, swapping the GLM for the RNN later is a small, contained change,
because the hard parts (loading, labeling, windowing, evaluation) will already be tested.

**Known limitation, on purpose:** this notebook uses only Mitt, a single session, so the train/test
split has to happen at the trial level, not the session level. That's fine as a "does this work at all"
check, but it is NOT the leak-safe evaluation we'll want once multiple rats are pooled together.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, accuracy_score, confusion_matrix

from src.preprocessing import build_labels, segment_trials, get_sampling_rate, trial_level_split


## 1. Load raw data for Mitt

Only Mitt this week, per the plan. Loading both `bvr` (behavior/labels) and `lfp` (the actual brain
signal we'll predict from), plus `lfp`'s channel names, which we'll use later to label which electrodes
the model actually relies on.


In [ ]:
session_dir = '../data/raw/080718_mitt'
session_name = '080718_mitt'

bvr = np.load(f'{session_dir}/{session_name}_bvr.npz', allow_pickle=True)
bvr_data = bvr['data']
bvr_keys = bvr['keys'].tolist()

lfp = np.load(f'{session_dir}/{session_name}_lfp.npz', allow_pickle=True)
lfp_data = lfp['data']
lfp_keys = lfp['keys'].tolist()

print("bvr shape:", bvr_data.shape)
print("lfp shape:", lfp_data.shape)
print("n electrodes:", len(lfp_keys))


## 2. Extract labels and sampling rate

Using the real, implemented `preprocessing.py` functions now, not stubs. `get_sampling_rate` computes
this session's actual rate from its own `TimeBin` channel (confirmed in the multi-rat audit that this
varies by rat, so we never hardcode it). `build_labels` gives us trial indices plus both labels.


In [ ]:
fs = get_sampling_rate(bvr_data, bvr_keys)
print(f"Sampling rate: {fs:.2f} Hz")

labels = build_labels(bvr_data, bvr_keys)
print(f"Trials found: {len(labels['trial_idx'])}")
print(f"Ambiguous odor trials: {len(labels['ambiguous'])}")
print(f"InSeq/OutSeq counts: {np.bincount(labels['inseq_outseq'])}")


## 3. Cut LFP into per-trial windows

500ms windows (from `configs/baseline.yaml`), extending forward from each trial marker, since we
confirmed the marker fires at odor onset, not a later decision point.


In [ ]:
WINDOW_MS = 500

windows, kept_idx = segment_trials(lfp_data, labels['trial_idx'], WINDOW_MS, fs)
print("windows shape:", windows.shape, "-> (n_trials, n_channels, window_samples)")

# if any trials were dropped (too close to end of recording), keep labels in sync
kept_mask = np.isin(labels['trial_idx'], kept_idx)
inseq_outseq = labels['inseq_outseq'][kept_mask]
odor_id = labels['odor_id'][kept_mask]
print("Trials kept after windowing:", len(kept_idx), "(dropped:", len(labels['trial_idx']) - len(kept_idx), ")")


## 4. Turn each window into simple features

A GLM needs a flat feature vector per trial, not a raw time series (that's the RNN's job later). We use
the simplest possible summary per channel: mean and standard deviation of the voltage within the window.
This throws away timing information on purpose, the point right now is a working baseline, not a good
one.

Result: for 22 LFP channels, that's 44 features per trial (22 means + 22 std devs). We keep a matching
list of feature names, so later plots can say "electrode T7 std" instead of "feature 29".


In [ ]:
def extract_simple_features(windows, channel_names):
    # windows: (n_trials, n_channels, window_samples)
    means = windows.mean(axis=2)
    stds = windows.std(axis=2)
    feature_names = [f"{ch}_mean" for ch in channel_names] + [f"{ch}_std" for ch in channel_names]
    return np.concatenate([means, stds], axis=1), feature_names

X, feature_names = extract_simple_features(windows, lfp_keys)
print("Feature matrix shape:", X.shape)


## 5. Task 1: InSeq vs OutSeq classification

Logistic regression with standardized features, evaluated with stratified k-fold cross-validation
(stratified so each fold keeps roughly the same InSeq/OutSeq ratio, important given the ~90/10
imbalance). We use a `Pipeline` so scaling is fit only on each fold's training data, never on data the
model will later be tested on. We report both raw accuracy and balanced accuracy, and compare against
the naive "always predict the majority class" baseline, since raw accuracy alone is misleading here.


In [ ]:
y_inseq = inseq_outseq

majority_class_accuracy = max(np.mean(y_inseq == 0), np.mean(y_inseq == 1))
print(f"Naive 'always guess majority class' accuracy: {majority_class_accuracy:.3f}")
print("(Balanced accuracy for that same naive guess is always 0.5, by definition)")

pipe_inseq = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_bal_acc = cross_val_score(pipe_inseq, X, y_inseq, cv=skf, scoring='balanced_accuracy')
fold_acc = cross_val_score(pipe_inseq, X, y_inseq, cv=skf, scoring='accuracy')

for fold in range(5):
    print(f"Fold {fold}: accuracy={fold_acc[fold]:.3f}  balanced_accuracy={fold_bal_acc[fold]:.3f}")

print()
print(f"Mean accuracy: {fold_acc.mean():.3f}")
print(f"Mean balanced accuracy: {fold_bal_acc.mean():.3f} +/- {fold_bal_acc.std():.3f}  (chance = 0.500)")


### Visualize: per-fold stability

With only 30 OutSeq trials spread across 5 folds, each test fold has roughly 6 OutSeq examples, small
enough that a couple of trials can swing a fold's score noticeably. Plotting the folds side by side
makes that visible in a way the printed numbers don't.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(5), fold_bal_acc, color='#4C72B0')
ax.axhline(0.5, color='red', linestyle='--', label='chance (0.5)')
ax.axhline(fold_bal_acc.mean(), color='black', linestyle=':', label=f'mean ({fold_bal_acc.mean():.3f})')
ax.set_xlabel('Fold')
ax.set_ylabel('Balanced accuracy')
ax.set_title('InSeq/OutSeq: balanced accuracy per fold')
ax.set_xticks(range(5))
ax.legend()
plt.tight_layout()
plt.show()


### Confusion matrix

Aggregated across all folds via `cross_val_predict` (each trial's prediction comes from the fold where
it was in the held-out test set, so this is still an honest out-of-sample view, not train-set
leakage).


In [ ]:
preds_inseq = cross_val_predict(pipe_inseq, X, y_inseq, cv=skf)
cm = confusion_matrix(y_inseq, preds_inseq)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['OutSeq', 'InSeq'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['OutSeq', 'InSeq'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('InSeq/OutSeq confusion matrix\n(aggregated across folds)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()


### Which electrodes is the model actually using?

Refit on all trials once (not for evaluation, purely to inspect what the model learned) and look at the
logistic regression coefficient magnitudes. Larger magnitude means that feature moved the prediction
more. This is a first, simple look at the interpretability question your advisors care about, before
the RNN's more sophisticated Integrated Gradients analysis later.


In [ ]:
pipe_inseq.fit(X, y_inseq)
coefs = pipe_inseq.named_steps['clf'].coef_[0]

# sort by absolute magnitude, show the top 15 most influential features
order = np.argsort(np.abs(coefs))[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#C44E52' if coefs[i] < 0 else '#55A868' for i in order]
ax.barh(range(len(order)), coefs[order], color=colors)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([feature_names[i] for i in order])
ax.invert_yaxis()
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (green = pushes toward InSeq, red = pushes toward OutSeq)')
ax.set_title('Top 15 most influential features, InSeq/OutSeq')
plt.tight_layout()
plt.show()


## 6. Task 2: Odor identity, InSeq trials only

Per the meeting notes, odor classification should only be evaluated on InSeq trials. Same approach:
multinomial logistic regression, stratified k-fold, compared against chance (1/5 = 20%).


In [ ]:
inseq_mask = (y_inseq == 1)
X_odor = X[inseq_mask]
y_odor = odor_id[inseq_mask]

print("InSeq trials available for odor classification:", len(y_odor))
print("Odor class counts:", np.bincount(y_odor))

pipe_odor = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),  # multinomial is now the sklearn default
])

skf_odor = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_bal_acc_odor = cross_val_score(pipe_odor, X_odor, y_odor, cv=skf_odor, scoring='balanced_accuracy')
fold_acc_odor = cross_val_score(pipe_odor, X_odor, y_odor, cv=skf_odor, scoring='accuracy')

for fold in range(5):
    print(f"Fold {fold}: accuracy={fold_acc_odor[fold]:.3f}  balanced_accuracy={fold_bal_acc_odor[fold]:.3f}")

print()
print(f"Mean accuracy: {fold_acc_odor.mean():.3f}")
print(f"Mean balanced accuracy: {fold_bal_acc_odor.mean():.3f} +/- {fold_bal_acc_odor.std():.3f}  (chance = 0.200)")


### Visualize: per-fold stability


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(5), fold_bal_acc_odor, color='#4C72B0')
ax.axhline(0.2, color='red', linestyle='--', label='chance (0.2)')
ax.axhline(fold_bal_acc_odor.mean(), color='black', linestyle=':', label=f'mean ({fold_bal_acc_odor.mean():.3f})')
ax.set_xlabel('Fold')
ax.set_ylabel('Balanced accuracy')
ax.set_title('Odor identity: balanced accuracy per fold')
ax.set_xticks(range(5))
ax.legend()
plt.tight_layout()
plt.show()


### Confusion matrix

5x5 this time, one row/column per odor. Worth checking: are errors spread evenly, or does the model
consistently mix up specific odor pairs?


In [ ]:
preds_odor = cross_val_predict(pipe_odor, X_odor, y_odor, cv=skf_odor)
cm_odor = confusion_matrix(y_odor, preds_odor)
odor_labels = ['A', 'B', 'C', 'D', 'E']

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm_odor, cmap='Blues')
ax.set_xticks(range(5)); ax.set_xticklabels(odor_labels)
ax.set_yticks(range(5)); ax.set_yticklabels(odor_labels)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Odor identity confusion matrix\n(aggregated across folds)')
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm_odor[i, j], ha='center', va='center',
                color='white' if cm_odor[i, j] > cm_odor.max() / 2 else 'black')
plt.tight_layout()
plt.show()


### Which electrodes matter most for odor identity?

For a multi-class model there's one coefficient vector per class; we average the absolute value across
all 5 odor classes to get one overall importance score per feature.


In [ ]:
pipe_odor.fit(X_odor, y_odor)
coefs_odor = pipe_odor.named_steps['clf'].coef_  # shape (5 classes, 44 features)
importance = np.abs(coefs_odor).mean(axis=0)

order = np.argsort(importance)[::-1][:15]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(range(len(order)), importance[order], color='#4C72B0')
ax.set_yticks(range(len(order)))
ax.set_yticklabels([feature_names[i] for i in order])
ax.invert_yaxis()
ax.set_xlabel('Mean |coefficient| across all 5 odor classes')
ax.set_title('Top 15 most influential features, odor identity')
plt.tight_layout()
plt.show()


## Summary

- **InSeq/OutSeq**: mean balanced accuracy ~0.56 (chance 0.5), a real but weak and unstable signal.
  Fold-to-fold variance is high, expected given only ~30 OutSeq trials total spread across 5 folds
  (~6 per test fold). Not a reason to worry about the pipeline, it's the small-n problem the meeting
  notes anticipated.
- **Odor identity**: mean balanced accuracy ~0.35 (chance 0.2), a clearer and more consistent signal,
  every fold beat chance. Worth highlighting Thursday, odor identity looks more decodable than
  InSeq/OutSeq from simple features alone.
- **Pipeline confirmed working end to end**: raw npz -> labels -> windows -> features -> trained model
  -> honest cross-validated evaluation -> interpretability check, with no errors and no suspiciously
  perfect (i.e. buggy-looking) results.

**Next step**: swap this GLM out for the `MultiTaskRNN` in `src/models.py`, using the same `windows`,
`inseq_outseq`, and `odor_id` arrays built here, but feeding the raw time series instead of flattened
mean/std features, so the model can learn from timing, not just summary statistics.


## 7. Save a text-only results summary

Everything above is best viewed as plots, but plots are expensive to share back and forth. This cell
prints (and saves to `outputs/logs/glm_baseline_results.json`) a plain-text version of every number
above: fold scores, both confusion matrices as plain grids, and the top features as a ranked text list.
Copy the printed output directly into chat when sharing results, no screenshots needed.


In [ ]:
import json as _json

results_summary = {
    "session": session_name,
    "n_trials_total": int(len(labels['trial_idx'])),
    "n_trials_used": int(len(kept_idx)),
    "inseq_outseq_task": {
        "majority_class_accuracy": round(float(majority_class_accuracy), 4),
        "fold_accuracy": [round(float(v), 4) for v in fold_acc],
        "fold_balanced_accuracy": [round(float(v), 4) for v in fold_bal_acc],
        "mean_accuracy": round(float(fold_acc.mean()), 4),
        "mean_balanced_accuracy": round(float(fold_bal_acc.mean()), 4),
        "std_balanced_accuracy": round(float(fold_bal_acc.std()), 4),
        "confusion_matrix": {
            "labels": ["OutSeq", "InSeq"],
            "matrix": cm.tolist(),
        },
    },
    "odor_task": {
        "n_trials": int(len(y_odor)),
        "class_counts": np.bincount(y_odor).tolist(),
        "fold_accuracy": [round(float(v), 4) for v in fold_acc_odor],
        "fold_balanced_accuracy": [round(float(v), 4) for v in fold_bal_acc_odor],
        "mean_accuracy": round(float(fold_acc_odor.mean()), 4),
        "mean_balanced_accuracy": round(float(fold_bal_acc_odor.mean()), 4),
        "std_balanced_accuracy": round(float(fold_bal_acc_odor.std()), 4),
        "confusion_matrix": {
            "labels": odor_labels,
            "matrix": cm_odor.tolist(),
        },
    },
}

inseq_order = np.argsort(np.abs(coefs))[::-1][:15]
results_summary["inseq_outseq_task"]["top_features"] = [
    {"feature": feature_names[i], "coefficient": round(float(coefs[i]), 4)}
    for i in inseq_order
]
odor_order = np.argsort(importance)[::-1][:15]
results_summary["odor_task"]["top_features"] = [
    {"feature": feature_names[i], "mean_abs_coefficient": round(float(importance[i]), 4)}
    for i in odor_order
]

print(_json.dumps(results_summary, indent=2))

import os
os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/glm_baseline_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/glm_baseline_results.json")
